# 🗓 Calendar Agent — Kaggle LoRA Training

Colab 대안. Kaggle은 별도 GPU quota를 제공 (주 30h, T4×2 또는 P100 16GB).

## 실행 전 준비 (한 번만)

### 1. Kaggle 설정 (우측 패널)
- **Accelerator**: `GPU T4 x2` 또는 `GPU P100` 선택
- **Internet**: `On` (외부 패키지·HF 모델 다운로드용)
- **Persistence**: `Variables and Files` (선택 — 세션 재시작 시 일부 보존)

### 2. 데이터 업로드 (Kaggle Dataset으로)
1. https://www.kaggle.com/datasets → `New Dataset`
2. 3개 파일 업로드:
   - `train.jsonl` (로컬 `D:\calenda\data\processed\train.jsonl`)
   - `val.jsonl` (동)
   - `golden.jsonl` (로컬 `D:\calenda\data\eval\golden.jsonl`)
3. Dataset 이름: `calendar-messages` (또는 임의)
4. Visibility: **Private**
5. Create

### 3. 이 노트북에 Dataset 첨부
- 우측 패널 `+ Add Input` → `Datasets` 탭 → 위에서 만든 dataset 검색 → 첨부
- 첨부되면 `/kaggle/input/datasets/sooryongbyun/calendar-messages/` 경로에 파일 보임

### 4. GitHub PAT 발급 (Private repo clone용)
https://github.com/settings/tokens/new?type=classic — repo scope, 7 days

---

준비 끝나면 셀을 위에서 아래로 순서대로 실행. 예상 시간: ~1시간.

# ✅ 1단계 · 준비 (Setup)

GPU·레포·설정·설치·데이터. **양자화만 할 때도 이 단계는 필수**(repo의 configs·scripts 필요).


## 1.1 GPU 확인

`Tesla T4` 또는 `Tesla P100` 표시되어야 함. 안 보이면 우측 패널 Accelerator 다시.

In [ ]:
!nvidia-smi

## 1.2 레포 clone (Private)

Kaggle은 working dir이 `/kaggle/working/`. 그 안에 clone.

In [ ]:
import os, getpass

%cd /kaggle/working
if not os.path.exists('Calenda'):
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('GITHUB_PAT')
        print('GITHUB_PAT: Kaggle Secrets 사용 OK')
    except Exception as _e:
        print(f'GITHUB_PAT Secret 못읽음({type(_e).__name__}) -> 직접 입력. '
              f'(Add-ons->Secrets에 라벨 GITHUB_PAT 등록 + 이 노트북에 attach 토글 ON)')
        token = getpass.getpass('GitHub PAT (repo scope): ').strip()
    !git clone https://{token}@github.com/sooryong/Calenda.git
    token = None
else:
    print('Calenda already cloned, force-syncing to origin/main…')
    !cd Calenda && git fetch origin && git reset --hard origin/main

%cd /kaggle/working/Calenda
!git log --oneline -1

## 1.3 라운드 설정 (CONFIG)

clone 직후 바로 현재 라운드를 표시한다. **라운드 변경은 이 셀의 `CONFIG` 한 줄만** 바꾸면 되고, 이후 사전점검·학습·평가 셀이 모두 이 `CONFIG`/`_cfg`/`_mcfg`를 재사용한다.


In [ ]:
# ── 라운드 설정 (CONFIG 한 줄로 라운드 결정 — 이후 모든 셀이 재사용) ──
#   0.5B(fallback): configs/train.yaml  /  Qwen3-0.6B: configs/train_qwen3_0_6b.yaml
CONFIG = 'configs/train_qwen3_0_6b.yaml'
import yaml
_cfg  = yaml.safe_load(open(CONFIG, encoding='utf-8'))
_mcfg = yaml.safe_load(open(_cfg['model_config'], encoding='utf-8'))
print(f"★ 현재 라운드 : {_cfg['run_name']}")
print(f"  베이스 모델 : {_mcfg['hf_id']}")
print(f"  output_dir : {_cfg['output_dir']}")
print(f"  유효 배치   : {_cfg['per_device_train_batch_size'] * _cfg['gradient_accumulation_steps']} (T4x2 DDP accum 자동분할)")


## 1.4 의존성 설치

Colab과 동일 — install + torchao 제거.

In [ ]:
# 설치. ⚠ Kaggle 사전설치 RAPIDS(cudf/cuml/dask-cuda)가 numba/cuda-core를 옛 버전에 고정해 둬서
#   "X requires ... which is incompatible" 충돌 경고가 뜨지만, 우리 학습은 RAPIDS 미사용 → 무해.
#   아래 grep으로 그 충돌 보고 줄만 숨김(실제 설치 실패 메시지는 패턴이 달라 그대로 보임).
!pip install -q -e .[train] 2>&1 | grep -vE "which is incompatible|pip's dependency resolver does not|source of the following dependency"
!pip uninstall torchao -y -q   # PEFT가 거부하는 구버전 torchao 제거 (LoRA엔 불필요)
import os, torch
os.environ["WANDB_DISABLED"] = "true"
# CUDA_VISIBLE_DEVICES 설정 안 함 — DDP(torchrun)가 2 GPU를 다 써야 함.
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(), "ngpu:", torch.cuda.device_count())

## 1.5 데이터 확인 (clone에 포함)


In [ ]:
# 데이터는 repo(git)에 포함 → clone으로 따라옴 (현 라운드 train + golden 50). 데이터셋 첨부 불필요.
for p in ["data/processed/train.jsonl", "data/processed/val.jsonl", "data/eval/golden.jsonl"]:
    n = sum(1 for _ in open(p, encoding="utf-8"))
    print(f"{p}: {n} records")

# 🎓 2단계 · 학습 + 평가 + 저장

사전점검 → 학습 → merge+골든평가 → HF 백업(어댑터+eval). **재양자화만 하려면 이 단계 건너뛰고 3단계로.**


## 2.1 학습 (데이터확인 · 사전점검 · 실행)

라운드는 위 **'1.3 라운드 설정'** 셀의 `CONFIG`로 결정된다. 아래 셀은 학습 직전 데이터 건수 최종 확인 → 이어서 학습 실행.

T4×2 DDP(torchrun)로 ~1시간. Kaggle 세션 한도 9시간 — 무관.


In [ ]:
# ── 학습 직전 최종 확인 (CONFIG/_cfg/_mcfg는 위 '1.3 라운드 설정' 셀에서 정의) ──
_n = sum(1 for l in open(_cfg['train_data'], encoding='utf-8') if l.strip())
print(f"★ 라운드 {_cfg['run_name']} | base {_mcfg['hf_id']} | {_cfg['train_data']}: {_n}건")


### 사전점검 — 학습/추론 프롬프트 정렬 확인

Qwen3는 thinking 모델이라 chat-template이 assistant 턴에 빈 `<think></think>`를 넣는다(**정상·의도된 설계**). 핵심은 *제거*가 아니라 **학습 렌더 == 추론 프롬프트 + gold JSON** 정렬이다 — `eval_model.infer`가 `enable_thinking=False`로 같은 빈 think를 프리필하므로 맞아야 한다.
'라운드 확인' 셀을 먼저 실행한 뒤 이 셀을 돌려라. `OK` 면 학습 진행, `assert` 멈추면 분포 불일치(템플릿 점검).


In [ ]:
# ── 사전점검: 학습 렌더 == 추론 프롬프트 + gold JSON (Qwen3 non-thinking 정렬) ──
#   '1.3 라운드 설정' 셀을 먼저 실행해 _mcfg를 채운 뒤 이 셀 실행.
from transformers import AutoTokenizer
import json
_tok = AutoTokenizer.from_pretrained(_mcfg['hf_id'], trust_remote_code=_mcfg.get('trust_remote_code', False))
_user = "<채널: KakaoTalk>\n<메시지>\n내일 3시 회의\n</메시지>"
_gold = json.dumps({'has_schedule': True, 'events': []}, ensure_ascii=False)
_sys  = _mcfg['system_prompt']
_full = [{'role':'system','content':_sys}, {'role':'user','content':_user}, {'role':'assistant','content':_gold}]
_pre  = [{'role':'system','content':_sys}, {'role':'user','content':_user}]
# 학습 렌더 (train_lora.to_chat와 동일)
_train = _tok.apply_chat_template(_full, tokenize=False, add_generation_prompt=False)
# 추론 렌더 (eval_model.infer와 동일: thinking 모델이면 enable_thinking=False)
_extra = {}
if 'enable_thinking' in (getattr(_tok, 'chat_template', None) or ''):
    _extra['enable_thinking'] = False
_infer = _tok.apply_chat_template(_pre, tokenize=False, add_generation_prompt=True, **_extra)
print('--- 학습 렌더 ---'); print(_train)
print('--- 추론 프롬프트 ---'); print(_infer)
_aligned = _train.startswith(_infer)
_tail = _train[len(_infer):] if _aligned else ''
_json_next = _tail.lstrip().startswith('{')
print()
print('정렬:', _aligned, '| 추론 직후 JSON:', _json_next, '->', 'OK: 학습 진행' if (_aligned and _json_next) else 'FAIL: 분포 불일치')
assert _aligned and _json_next, '학습 렌더가 (추론 프롬프트 + gold JSON)와 불일치 — train/eval 프롬프트 정렬 깨짐'


In [ ]:
# ── 학습 실행 (위 '1.3 라운드 설정' 셀 먼저 실행해 CONFIG 설정) ──
# DDP: T4 2개를 진짜로 병렬 사용 (torchrun, DataParallel 아님)
print(f"학습 시작 → {CONFIG} (라운드: {_cfg['run_name']})")
!torchrun --nproc_per_node=2 scripts/train_lora.py --config {CONFIG}

## 2.2 평가 (merge + 골든셋)

merge 후 골든셋으로 바로 평가해 `time_match_rate`/`final_score`를 확인한다. 로컬 CPU 평가(약 14분)보다 GPU가 훨씬 빠르다. 결과 json은 아래 6번 zip에 포함된다.

In [ ]:
# CONFIG에서 모든 경로 자동 인식 — base는 model_config의 hf_id에서 읽음.
import yaml, os
_cfg = yaml.safe_load(open(CONFIG, encoding='utf-8'))
_mcfg = yaml.safe_load(open(_cfg['model_config'], encoding='utf-8'))
BASE = _mcfg['hf_id']                          # model_config의 hf_id 자동
LORA_DIR = _cfg['output_dir']                  # 예: models/lora/r12-qwen0.5b
NAME = os.path.basename(LORA_DIR)              # r12-qwen0.5b
MERGED_DIR = f'models/merged/{NAME}'
EVAL_JSON = f'logs/eval_{NAME}.json'
GOLDEN = _cfg.get('eval_golden', 'data/eval/golden.jsonl')
ZIP = f'/kaggle/working/lora_{NAME}.zip'       # 모델별로 구분
print(f'base={BASE}\nlora={LORA_DIR}\ngolden={GOLDEN}\nzip={ZIP}')

# merge → 실 골든셋 평가 (Kaggle GPU)
!python scripts/merge_lora.py --base {BASE} --lora {LORA_DIR} --out {MERGED_DIR}
!python scripts/eval_model.py --model {MERGED_DIR} --eval {GOLDEN} --out {EVAL_JSON}
print(f'--- {EVAL_JSON} ---')
print(open(EVAL_JSON, encoding='utf-8').read())

## 2.3 HF 백업 (어댑터 + eval)

Kaggle 세션은 시간이 지나면 사라진다. 학습+eval 직후 **어댑터(40MB)와 eval 결과를 HF private repo에 백업**하면, 나중에 3-0에서 불러와 재학습 없이 3단계(양자화·검증)을 돌릴 수 있다.
- 인증: Kaggle **Secrets `HF_TOKEN`**(Add-ons→Secrets) 우선, 없으면 getpass 입력. **토큰은 코드/채팅에 안 남는다.**


In [ ]:
# ── 2.3 (선택) HF 백업: 어댑터(40MB)+eval, (옵션)merged(1.2GB) → HF private repo ──
HF_REPO = 'sooryong9885/Calenda-Qwen3-0.6B'
DO_BACKUP = True        # 어댑터+eval 백업
BACKUP_MERGED = False   # True면 merged도(1.2GB) -> NAME/merged/ (재양자화 시 merge 생략)
if DO_BACKUP:
    import os
    try:
        from kaggle_secrets import UserSecretsClient
        _tok = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        from getpass import getpass; _tok = getpass('HF write token: ')
    from huggingface_hub import HfApi, login
    login(token=_tok)
    api = HfApi()
    api.create_repo(HF_REPO, repo_type='model', private=True, exist_ok=True)
    api.upload_folder(folder_path=LORA_DIR, repo_id=HF_REPO, path_in_repo=NAME,
                      repo_type='model', commit_message=f'{NAME} adapter')
    if os.path.isfile(EVAL_JSON):
        api.upload_file(path_or_fileobj=EVAL_JSON, path_in_repo=f'{NAME}/eval.json',
                        repo_id=HF_REPO, repo_type='model')
    if BACKUP_MERGED and os.path.isdir(MERGED_DIR):
        api.upload_folder(folder_path=MERGED_DIR, repo_id=HF_REPO, path_in_repo=f'{NAME}/merged',
                          repo_type='model', commit_message=f'{NAME} merged')
        print('merged도 업로드 -> NAME/merged/')
    print(f'백업 완료 -> https://huggingface.co/{HF_REPO}/tree/main/{NAME}')
else:
    print('DO_BACKUP=False -> 백업 건너뜀')


## 2.4 결과 패키징 (zip 다운로드)

Kaggle은 `/kaggle/working/` 안의 파일을 노트북 우측 패널 `Output`에서 다운로드 가능.
zip을 `/kaggle/working/` 루트에 두면 됨.

In [ ]:
!ls -la {LORA_DIR}/
!zip -r {ZIP} {LORA_DIR} {CONFIG} configs/lora.yaml {_cfg['model_config']} {EVAL_JSON}
!ls -la {ZIP}

### 다운로드 방법

**FileLink는 Kaggle 프록시에서 동작하지 않는다(404 남).** 아래 방법을 쓸 것:

1. **우상단 `Save Version` → `Quick Save`** (재실행 안 함, 현재 `/kaggle/working` 스냅샷) → 저장되면 버전 열기 → **Output** 탭에서 `lora_<라운드>.zip` 다운로드  ← 가장 확실
2. 또는 에디터 우측 **Output(`/kaggle/working`) 파일 패널**에서 `lora_<라운드>.zip` 다운로드 (안 보이면 새로고침)
3. 또는 로컬에서 Kaggle API: `kaggle kernels output <user>/<notebook-slug> -p .`

학습 직후 평가(아래 5.5)를 돌렸다면 `logs/eval_<라운드>-qwen.json`의 `time_match_rate` / `final_score`를 셀 출력에서 바로 확인할 수 있다.

In [ ]:
# 참고: FileLink는 Kaggle에서 종종 404. 위 '다운로드 방법'의 Quick Save -> Output 사용 권장.
from IPython.display import FileLink
FileLink(ZIP)

# 📦 3단계 · 양자화 + 평가 + 저장

merged 확보 → 양자화(Q4/Q8) → 골든평가 → HF 저장(GGUF+manifest).


## 3. 양자화 파이프라인 (merged 확보 → 양자화 → 평가 → HF 저장)

⚠️ **양자화만 하는 새 세션이라도 먼저 1단계 준비(clone·설치)를 실행**해야 한다 (3-0~3-C가 repo의 `configs/`·`scripts/`를 사용). 안 하면 `레포 없음` assert로 멈춘다.

**3-0** merged 확보: 로컬에 있으면 그대로, 없으면 **HF의 라운드 목록을 표로 보여주고**(점수·보유 아티팩트 포함) 번호나 라운드명으로 골라 다운로드→merge한다. (repo는 고정, 무엇을 받을지는 목록 보고 결정)
→ **3-A** 양자화(Q4_K_M·Q8_0) → **3-B** 양자화 평가(FP16 대비) → **3-C** 양자화 결과 HF 저장.


In [ ]:
# ── 3-0. merged 확보: 로컬 있으면 사용, 없으면 HF 라운드 목록에서 선택 -> 로딩 ──
import os, glob, shutil, json, yaml
# 레포 루트 보장 (Section 1 clone이 cwd를 잡지만, 커널 재시작 등으로 풀렸으면 복구)
_REPO = '/kaggle/working/Calenda'
if not os.path.isfile('configs/train_qwen3_0_6b.yaml') and os.path.isdir(_REPO):
    os.chdir(_REPO)
assert os.path.isfile('configs/train_qwen3_0_6b.yaml'), '레포 없음 - 먼저 1단계 준비(clone·설치) 실행'
HF_REPO = 'sooryong9885/Calenda-Qwen3-0.6B'   # 고정
_cfg  = yaml.safe_load(open('configs/train_qwen3_0_6b.yaml', encoding='utf-8'))
_mcfg = yaml.safe_load(open(_cfg['model_config'], encoding='utf-8'))
NAME = globals().get('NAME') or os.path.basename(_cfg['output_dir'])
MERGED_DIR = globals().get('MERGED_DIR') or f'models/merged/{NAME}'
GOLDEN = _cfg.get('eval_golden', 'data/eval/golden.jsonl')
EVAL_JSON = f'logs/eval_{NAME}.json'
def _has_merged(p): return os.path.isdir(p) and bool(glob.glob(f'{p}/*.safetensors'))
if _has_merged(MERGED_DIR):
    print(f'OK merged 있음 -> {MERGED_DIR} (그대로 3-A 양자화)')
else:
    print(f'merged 없음: {MERGED_DIR} -> HF에서 선택')
    try:
        from kaggle_secrets import UserSecretsClient
        _tok = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        from getpass import getpass; _tok = getpass('HF token: ')
    from huggingface_hub import HfApi, login, snapshot_download, hf_hub_download
    login(token=_tok)
    api = HfApi()
    _files = api.list_repo_files(HF_REPO, repo_type='model')
    _rounds = sorted({f.split('/')[0] for f in _files if '/' in f and not f.startswith('.')})
    if not _rounds:
        raise SystemExit('HF repo에 라운드 없음')
    print()
    print('HF 라운드 목록 (아티팩트 / 점수):')
    for i, r in enumerate(_rounds):
        rf = [f for f in _files if f.startswith(r + '/')]
        flags = []
        if any('adapter_model' in f for f in rf): flags.append('adapter')
        if any(f.startswith(r + '/merged/') for f in rf): flags.append('merged')
        if any(f.endswith('.gguf') for f in rf): flags.append('gguf')
        sc = ''
        if (r + '/eval.json') in _files:
            try:
                _ev = json.load(open(hf_hub_download(HF_REPO, r + '/eval.json', repo_type='model'), encoding='utf-8'))
                sc = f"  final={_ev.get('final_score', 0):.3f}"
            except Exception:
                pass
        print(f"  [{i}] {r}  ({'/'.join(flags) or '-'}){sc}")
    print()
    _sel = input('번호 또는 라운드명 입력 (취소=n): ').strip()
    if _sel.lower() == 'n':
        raise SystemExit('취소됨')
    NAME = _rounds[int(_sel)] if _sel.isdigit() else _sel
    MERGED_DIR = f'models/merged/{NAME}'
    EVAL_JSON = f'logs/eval_{NAME}.json'
    _l = snapshot_download(repo_id=HF_REPO, repo_type='model', allow_patterns=f'{NAME}/*')
    _base = f'{_l}/{NAME}'
    if os.path.isfile(f'{_base}/eval.json'):
        os.makedirs('logs', exist_ok=True); shutil.copy(f'{_base}/eval.json', EVAL_JSON)
    if _has_merged(f'{_base}/merged'):
        shutil.copytree(f'{_base}/merged', MERGED_DIR, dirs_exist_ok=True)
        print(f'OK HF merged 복사 -> {MERGED_DIR}')
    elif glob.glob(f'{_base}/adapter_model.safetensors'):
        !python scripts/merge_lora.py --base {_mcfg['hf_id']} --lora {_base} --out {MERGED_DIR}
        print(f'OK HF 어댑터 -> merge -> {MERGED_DIR}')
    else:
        print('주의: 이 라운드엔 adapter/merged 없음(gguf만?) - 3-A 불가, 3-B 검증/배포만')
print('NAME=', NAME, '| MERGED_DIR=', MERGED_DIR, '| FP16 기준:', os.path.isfile(EVAL_JSON))


### 3-A 양자화 (merged → Q4_K_M · Q8_0)

3-0에서 확보한 `MERGED_DIR`을 GGUF로 변환·양자화한다. (llama.cpp 빌드 ~5분, 처음 1회)


In [ ]:
# ── 3-A. 양자화: merged → GGUF Q4_K_M + Q8_0 (진행 표시) ──
import os
LLAMA = '/kaggle/working/llama.cpp'
GGUF_DIR = f'models/gguf/{NAME}'; os.makedirs(GGUF_DIR, exist_ok=True)
QUANT = f'{LLAMA}/build/bin/llama-quantize'
print('[1/4] llama.cpp 소스 + gguf 패키지')
if not os.path.isdir(LLAMA):
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp.git {LLAMA}
!pip install -q gguf
if not os.path.isfile(QUANT):
    print('[2/4] cmake 빌드 (~5분, 진행률 %로 표시)')
    !cmake -S {LLAMA} -B {LLAMA}/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /tmp/llama_cfg.log 2>&1
    !cmake --build {LLAMA}/build -j --target llama-quantize 2>&1 | grep --line-buffered -E "[0-9]+%|Built target|rror"
else:
    print('[2/4] 빌드 스킵 (바이너리 있음)')
print('   quantize 바이너리:', os.path.isfile(QUANT))
print('[3/4] merged -> f16 GGUF 변환')
F16 = f'{GGUF_DIR}/{NAME}.f16.gguf'
!python {LLAMA}/convert_hf_to_gguf.py {MERGED_DIR} --outfile {F16} --outtype f16
print('[4/4] 양자화 (Q4_K_M, Q8_0)')
for q in ['Q4_K_M', 'Q8_0']:
    print(f'   -> {q} ...')
    !{QUANT} {F16} {GGUF_DIR}/{NAME}.{q}.gguf {q}
!cp {GGUF_DIR}/{NAME}.Q4_K_M.gguf {GGUF_DIR}/{NAME}.Q8_0.gguf /kaggle/working/
print('완료 — 파일:')
!ls -la {GGUF_DIR}


In [ ]:
# ── 3-B. (선택) GGUF 골든 검증: Q4_K_M vs Q8_0 vs FP16(있으면) ──
!pip install -q llama-cpp-python
import json, os
GGUF_DIR = f'models/gguf/{NAME}'
res = {}
for q in ['Q8_0', 'Q4_K_M']:
    outj = f'logs/eval_{NAME}.{q}.json'
    !python scripts/eval_gguf.py --gguf {GGUF_DIR}/{NAME}.{q}.gguf --eval {GOLDEN} --out {outj} --failures_out data/failures/{NAME}.{q}.fail.jsonl --n_gpu_layers 0
    res[q] = json.load(open(outj, encoding='utf-8'))
fp16 = json.load(open(EVAL_JSON, encoding='utf-8')) if os.path.isfile(EVAL_JSON) else None
KEYS = ['final_score','json_valid_rate','title_f1_avg','time_match_rate','location_f1_avg','has_schedule_acc','event_count_acc']
print()
if fp16:
    print(f"{'metric':<20}{'FP16':>9}{'Q8_0':>9}{'Q4_K_M':>9}")
    for k in KEYS:
        print(f"{k:<20}{fp16[k]:>9.3f}{res['Q8_0'][k]:>9.3f}{res['Q4_K_M'][k]:>9.3f}")
else:
    print('(FP16 기준 없음 → Q8_0 vs Q4_K_M만)')
    print(f"{'metric':<20}{'Q8_0':>9}{'Q4_K_M':>9}")
    for k in KEYS:
        print(f"{k:<20}{res['Q8_0'][k]:>9.3f}{res['Q4_K_M'][k]:>9.3f}")


### 3-C GGUF + 버전정보(manifest) HF 업로드

양자화·검증 후 **GGUF(Q4_K_M·Q8_0)와 `manifest.json`(버전정보)을 HF에 업로드**한다. manifest엔 라운드·베이스·git커밋·각 GGUF의 크기/md5·점수(FP16/Q8/Q4)·생성시각이 들어간다.
→ 이후 폰 재배포/새 기기는 3-0에서 **GGUF만 받아** merge·양자화 없이 끝. (3-0 `PREFER_GGUF=True`)


In [ ]:
# ── 3-C. (선택) GGUF + manifest(버전정보) HF 업로드 ──
import os, json, hashlib, subprocess, datetime
HF_REPO = 'sooryong9885/Calenda-Qwen3-0.6B'
GGUF_DIR = f'models/gguf/{NAME}'
def _md5(p, chunk=1<<20):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
try:
    GIT = subprocess.check_output(['git','rev-parse','--short','HEAD']).decode().strip()
except Exception:
    GIT = None
quants = {}
for q in ['Q4_K_M', 'Q8_0']:
    p = f'{GGUF_DIR}/{NAME}.{q}.gguf'
    if os.path.isfile(p):
        quants[q] = {'file': os.path.basename(p), 'size_mb': round(os.path.getsize(p)/1048576, 1), 'md5': _md5(p)}
scores = {}
if os.path.isfile(EVAL_JSON):
    scores['FP16'] = json.load(open(EVAL_JSON, encoding='utf-8'))
try:
    scores.update(res)   # 3-B 결과(Q8_0/Q4_K_M) 있으면 포함
except NameError:
    pass
manifest = {
    'round': NAME,
    'base': _mcfg['hf_id'],
    'git_commit': GIT,
    'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'quants': quants,
    'scores_final': {k: round(v.get('final_score', 0), 4) for k, v in scores.items()},
}
os.makedirs(GGUF_DIR, exist_ok=True)
json.dump(manifest, open(f'{GGUF_DIR}/manifest.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print(json.dumps(manifest, ensure_ascii=False, indent=2))
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    from getpass import getpass; _tok = getpass('HF write token: ')
from huggingface_hub import HfApi, login
login(token=_tok)
api = HfApi()
api.create_repo(HF_REPO, repo_type='model', private=True, exist_ok=True)
_msg = f'{NAME} gguf+manifest @ {GIT}'
for q, info in quants.items():
    api.upload_file(path_or_fileobj=f"{GGUF_DIR}/{info['file']}", path_in_repo=f"{NAME}/{info['file']}", repo_id=HF_REPO, repo_type='model', commit_message=_msg)
api.upload_file(path_or_fileobj=f'{GGUF_DIR}/manifest.json', path_in_repo=f'{NAME}/manifest.json', repo_id=HF_REPO, repo_type='model', commit_message=_msg)
print(f'업로드 완료 -> https://huggingface.co/{HF_REPO}/tree/main/{NAME}')
